# Figure 5 — Chemical space of reagents (t-SNE)

This notebook reproduces the t-SNE projection of the reagent chemical space across the curated dataset of asymmetric organocatalytic Mannich reactions. Unique SMILES from the `nucleophile` and `electrophile` columns are embedded together into a shared two-dimensional space.

## Method summary

- **Representation**: chirality-aware Morgan fingerprints (radius 2, 2,048 bits)
- **Pooling**: unique SMILES from both reagent columns, deduplicated
- **Embedding**: t-SNE with the Jaccard metric (= 1 − Tanimoto on binary bits)
- **Coloring**: blue for nucleophiles, red for electrophiles

## Outputs

- `figure_05_chemical_space.pdf` — vector, primary submission format
- `figure_05_chemical_space.svg` — vector, editable
- `figure_05_chemical_space.png` — raster, 300 dpi

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.manifold import TSNE

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

## 2. Matplotlib style

Consistent with the rest of the manuscript figures (Arial, no top/right spines, editable text in PDF/SVG).

In [ ]:
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':          9,
    'axes.labelsize':    10,
    'axes.titlesize':    10,
    'axes.linewidth':     0.8,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'legend.fontsize':    9,
    'legend.frameon':     True,
    'figure.dpi':         120,
    'savefig.dpi':        300,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.05,
    'pdf.fonttype':       42,
    'ps.fonttype':        42,
    'svg.fonttype':      'none',
})

COLOR_NUCLEOPHILE  = '#4A90C7'  # blue
COLOR_ELECTROPHILE = '#C0392B'  # red

## 3. Load the dataset and collect unique SMILES

Unique SMILES are extracted independently from the `nucleophile` and `electrophile` columns. The two lists are then concatenated for joint embedding. Any SMILES that appears in both columns (chemically possible, though rare) is treated as a single deduplicated point and assigned to the column it first appears in.

In [ ]:
DATA_PATH = 'Mannich_dataset.csv'  # adjust path as needed
df = pd.read_csv(DATA_PATH)

nuc_unique = df['nucleophile'].dropna().astype(str).unique().tolist()
ele_unique = df['electrophile'].dropna().astype(str).unique().tolist()

# Deduplicate across the two pools, keeping the first assignment
seen = set()
rows = []
for smi in nuc_unique:
    if smi not in seen:
        seen.add(smi)
        rows.append((smi, 'nucleophile'))
for smi in ele_unique:
    if smi not in seen:
        seen.add(smi)
        rows.append((smi, 'electrophile'))

reagents = pd.DataFrame(rows, columns=['smiles', 'role'])
print(f'Unique nucleophiles:  {len(nuc_unique)}')
print(f'Unique electrophiles: {len(ele_unique)}')
print(f'Combined (deduped):   {len(reagents)}')

## 4. Morgan fingerprints

Each unique SMILES is converted to a 2,048-bit Morgan fingerprint with radius 2 and chirality encoded.

In [ ]:
FP_RADIUS = 2
FP_NBITS  = 2048

def morgan_fp(smiles, radius=FP_RADIUS, n_bits=FP_NBITS):
    """Return a chirality-aware Morgan fingerprint as a numpy uint8 array,
    or None if the SMILES cannot be parsed."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    bitvect = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius=radius, nBits=n_bits, useChirality=True)
    arr = np.zeros((n_bits,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(bitvect, arr)
    return arr

reagents['fp'] = reagents['smiles'].apply(morgan_fp)
reagents = reagents[reagents['fp'].notna()].reset_index(drop=True)
X = np.vstack(reagents['fp'].values)
print(f'Fingerprint matrix: {X.shape}')

## 5. t-SNE embedding

Both reagent pools are embedded jointly so that the two groups live in the same coordinate system and can be compared directly. The Jaccard metric is appropriate for binary fingerprints (equivalent to 1 − Tanimoto). A fixed `random_state` ensures the layout is reproducible.

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    metric='jaccard',
    random_state=42,
    init='random',
)
embedding = tsne.fit_transform(X)
reagents['x'] = embedding[:, 0]
reagents['y'] = embedding[:, 1]

## 6. Plot

Single panel. Nucleophiles are drawn first (in blue); electrophiles are drawn on top (in red) so the smaller of the two groups remains visible where the two clouds overlap. The legend is anchored inside the axes (upper right corner). The two axes are labeled `t-SNE-1` and `t-SNE-2`; note that, by the nature of t-SNE, only relative distances between points are meaningful — absolute coordinate values depend on the choice of random seed and hyperparameters.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 5.4))

nuc = reagents[reagents['role'] == 'nucleophile']
ele = reagents[reagents['role'] == 'electrophile']

ax.scatter(nuc['x'], nuc['y'],
           s=18, color=COLOR_NUCLEOPHILE, alpha=0.75,
           edgecolors='white', linewidths=0.3, label='nucleophiles')
ax.scatter(ele['x'], ele['y'],
           s=18, color=COLOR_ELECTROPHILE, alpha=0.75,
           edgecolors='white', linewidths=0.3, label='electrophiles')

ax.set_xlabel('t-SNE-1')
ax.set_ylabel('t-SNE-2')
ax.tick_params(axis='both', which='major', labelsize=8)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

# Legend anchored inside the plot
legend = ax.legend(
    loc='upper right',
    frameon=True,
    framealpha=0.92,
    edgecolor='lightgrey',
    handletextpad=0.4,
    borderpad=0.6,
)
legend.get_frame().set_linewidth(0.5)

plt.show()

## 7. Export

In [ ]:
for ext in ('pdf', 'svg', 'png'):
    fig.savefig(f'figure_05_chemical_space.{ext}',
                dpi=300 if ext == 'png' else None)